# Golf Forecasting

Generate a forecasting dataset about professional golf (tournaments, majors, rankings) using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments.

In [1]:
%pip install lightningrod-ai python-dotenv pandas openai

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [2]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for golf forecasting.

In [3]:
instructions = """
Generate binary forecasting questions about professional golf across all major tours and events.

Cover what golf fans bet on: tournament outcomes, cuts, matchups, majors, team events, season races, world rankings, and player milestones.

Questions should be specific, verifiable, and span the full probability spectrum.
"""

good_examples = [
    "Will Scottie Scheffler win the 2025 Masters?",
    "Will the 2025 US Open winning score be under par?",
    "Will Tiger Woods make the cut at the 2025 Masters?",
    "Will Rory McIlroy finish top 5 at the 2025 US Open?",
    "Will any LIV player win a major championship in 2025?",
    "Will Europe win the 2025 Ryder Cup?",
    "Will any player win 4+ PGA Tour events in 2025?",
    "Will Scottie Scheffler remain world #1 through June 2025?",
    "Will a first-time major winner emerge at the 2025 PGA Championship?",
    "Will Nelly Korda win the 2025 US Women's Open?",
]

bad_examples = [
    "Will someone win the tournament? (obvious)",
    "Will golf be exciting? (subjective)",
    "Will there be birdies? (trivial)",
]

search_queries = [
    "PGA Tour",
    "LIV Golf",
    "LPGA",
    "golf major championship",
    "Ryder Cup Presidents Cup",
    "golf world rankings",
    "professional golf",
    "women's golf",
    "European Tour golf",
]

In [4]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2024, 6, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=14,
        search_query=search_queries,
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=5,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=3,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_questions` to limit the run for testing.

In [5]:
dataset = lr.transforms.run(pipeline, max_questions=1000, name="Golf forecasting")
samples = dataset.download()

pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $47.67                                                                                           │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons  ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ NewsSeedGenerator… │ Complete             │  20 │ 200 │        0 │      0 │ -                  │      19s │  │
│  │ ForwardLookingQue… │ Complete             │ 200 │ 965 │       33 │      0 │ date_close not     │      15s │  │
│  │                    │                      │     │     │          │        │ after event_date   │          │  │
│  │                    │                      │     │     │          │        │ (33)               │          │  │
│  │ WebSearchLabelerT… │ Complete             │ 965 │ 796 │      169 │      0 │ Undetermined label │    1m 8s │  │
│  │                    │                      │     │     │          │        │ (164), Resolution  │          │  │
│  │                    │                      │     │     │          │        │ date is before     │          │  │
│  │                    │                      │     │     │          │        │ seed creation date │          │  │
│  │                    │                      │     │     │          │        │ (5)                │          │  │
│  │ NewsContextGenera… │ Complete             │ 796 │ 795 │        1 │      0 │ <failed_attempts>  │  11m 37s │  │
│  │                    │                      │     │     │          │        │                    │          │  │
│  │                    │                      │     │     │          │        │ <generation        │          │  │
│  │                    │                      │     │     │          │        │ number="1">        │          │  │
│  │                    │                      │     │     │          │        │ <exception>        │          │  │
│  │                    │                      │     │     │          │        │     Request timed  │          │  │
│  │                    │                      │     │     │          │        │ out.               │          │  │
│  │                    │                      │     │     │          │        │ </exception>       │          │  │
│  │                    │                      │     │     │          │        │ <completion>       │          │  │
│  │                    │                      │     │     │          │        │     None           │          │  │
│  │                    │                      │     │     │          │        │ </completion>      │          │  │
│  │                    │                      │     │     │          │        │ </generation>      │          │  │
│  │                    │                      │     │     │          │        │                    │          │  │
│  │                    │                      │     │     │          │        │ <generation        │          │  │
│  │                    │                      │     │     │          │        │ number="2">        │          │  │
│  │                    │                      │     │     │          │        │ <exception>        │          │  │
│  │                    │                      │     │  

948 samples (78.6% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. We filter by `date_close <= today` to only include questions that have already resolved.

In [ ]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(days_to_resolution_range=(1, None)),
    split=SplitParams(test_size=0.2),
)

for name, ds in [("Train", train_dataset), ("Test", test_dataset)]:
    data = ds.flattened()
    yes_count = sum(1 for s in data if s.get("label") in (1, "1", 1.0))
    print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    display(pd.DataFrame(data).head())

Train: 350 rows, 34.0% yes


,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,label,answer_type,label_confidence,...,reasoning,answer_sources,seed_text,seed_url,seed_creation_date,seed_search_query,context,meta_sample_id,meta_parent_sample_id,meta_processing_time_ms
0,23a607a2-e9db-45a9-8e33-67cd16a32b56,True,Will the Eastern Michigan University women's g...,2025-05-10T00:00:00,2024-07-15T00:00:00,The question resolves to 'Yes' if Eastern Mich...,2024-07-15T00:00:00,0,binary,1.00,...,The Eastern Michigan University (EMU) women's ...,https://vertexaisearch.cloud.google.com/ground...,Eastern Michigan Athletics\nCaterina Don Named...,https://emueagles.com/news/2024/7/9/womens-gol...,2024-07-15T00:00:00,women's golf,"[{'rendered_context': '', 'search_query': 'Eas...",fa146afa-b53f-48c1-8d2d-a81ce2dec41b,0107be94-88d9-4068-a355-ec38b8691376,844641.292
1,2736b5ea-b6b0-4fde-a237-96f6a3d9ee86,True,Will an Arizona Wildcats player be named the B...,2025-05-01T00:00:00,2024-07-15T00:00:00,The question resolves to 'Yes' if the Big 12 C...,2024-07-15T00:00:00,1,binary,1.00,...,The Arizona Wildcats officially joined the Big...,https://vertexaisearch.cloud.google.com/ground...,"TUCSON, Ariz. – Arizona Women's Golf Head Coac...",https://arizonawildcats.com/news/2024/7/15/bra...,2024-07-15T00:00:00,women's golf,[{'rendered_context': '--- ARTICLES [1] Arizon...,ece95e4d-151b-4af8-936a-5c7e18276b97,f7429e77-a524-46dd-ae47-821794e79938,988488.435
2,2eae4276-f449-45e9-8973-e760b6d36d61,True,Will Caterina Don remain in her role as the As...,2025-05-31T00:00:00,2024-07-15T00:00:00,The question resolves to 'Yes' if Caterina Don...,2024-07-15T00:00:00,1,binary,0.95,...,Caterina Don was hired as the first full-time ...,https://vertexaisearch.cloud.google.com/ground...,Eastern Michigan Athletics\nCaterina Don Named...,https://emueagles.com/news/2024/7/9/womens-gol...,2024-07-15T00:00:00,women's golf,"[{'rendered_context': '', 'search_query': 'Cat...",4fbfad5d-b425-4cfe-b04d-1db1279c80a8,0107be94-88d9-4068-a355-ec38b8691376,485377.689
3,35b2be70-0b7c-4b1b-b82f-2e3f0d3b61d8,True,Will the University of North Carolina women's ...,2025-04-15T00:00:00,2024-07-15T00:00:00,The question resolves to Yes if the UNC women'...,2024-07-15T00:00:00,1,binary,1.00,...,The University of North Carolina women's golf ...,https://vertexaisearch.cloud.google.com/ground...,University of North Carolina Athletics\nNeff's...,https://goheels.com/news/2024/7/15/neffs-contr...,2024-07-15T00:00:00,women's golf,[{'rendered_context': '--- ARTICLES [1] 2024-2...,05a3aff2-dee2-4111-a0f4-c085ba138679,c9839639-2fd2-4f0a-ad52-4566bbde89be,1052828.897
4,382c5c2e-db8f-4b8c-bbe8-6fc4e55edf53,True,Will the University of Arizona Women's Golf te...,2025-04-15T00:00:00,2024-07-15T00:00:00,The question resolves to 'Yes' if the Universi...,2024-07-15T00:00:00,1,binary,1.00,...,The University of Arizona Women's Golf team wo...,https://vertexaisearch.cloud.google.com/ground...,"TUCSON, Ariz. – Arizona Women's Golf Head Coac...",https://arizonawildcats.com/news/2024/7/15/bra...,2024-07-15T00:00:00,women's golf,[{'rendered_context': '--- ARTICLES [1] Arizon...,1fd71efb-9506-4c76-b1b6-d8eed1a30ac7,f7429e77-a524-46dd-ae47-821794e79938,1089828.234


Test: 143 rows, 32.2% yes


,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,label,answer_type,label_confidence,...,reasoning,answer_sources,seed_text,seed_url,seed_creation_date,seed_search_query,context,meta_sample_id,meta_parent_sample_id,meta_processing_time_ms
0,69d062b4-681c-431f-9e01-e13befba3ea0,True,Will Luke Clanton finish in the top 10 of the ...,2025-07-07T00:00:00,2025-06-24T00:00:00,The question resolves to 'Yes' if Luke Clanton...,2025-06-24T00:00:00,0,binary,1.0,...,Luke Clanton participated in the 2025 John Dee...,https://vertexaisearch.cloud.google.com/ground...,"Title: No. 15 Ben Griffin, rising star Luke Cl...",https://www.wqad.com/article/sports/john-deere...,2025-06-24T00:00:00,golf world rankings,"[{'rendered_context': '', 'search_query': 'Luk...",7cdbf935-029e-4ce5-bcb4-d6cbf016303a,c96f580b-8dff-42a0-90dc-3e5c888680c6,489996.714
1,c3ef9b69-79e5-42c4-a26c-b0d02bf82abb,True,Will Ben Griffin be ranked in the top 10 of th...,2025-07-07T00:00:00,2025-06-24T00:00:00,The question resolves to 'Yes' if Ben Griffin'...,2025-06-24T00:00:00,0,binary,1.0,...,Ben Griffin was ranked No. 17 in the Official ...,https://vertexaisearch.cloud.google.com/ground...,"Title: No. 15 Ben Griffin, rising star Luke Cl...",https://www.wqad.com/article/sports/john-deere...,2025-06-24T00:00:00,golf world rankings,"[{'rendered_context': '', 'search_query': 'Ben...",c3a577ab-069d-4857-8c52-9fce6f44e876,c96f580b-8dff-42a0-90dc-3e5c888680c6,493772.150
2,e1d9f94a-8fa0-4caf-a85b-2f2602e7e9ae,True,Will Luke Clanton outscore Ben Griffin in the ...,2025-07-04T00:00:00,2025-06-24T00:00:00,The question resolves to 'Yes' if Luke Clanton...,2025-06-24T00:00:00,1,binary,1.0,...,The first round of the 2025 John Deere Classic...,https://vertexaisearch.cloud.google.com/ground...,"Title: No. 15 Ben Griffin, rising star Luke Cl...",https://www.wqad.com/article/sports/john-deere...,2025-06-24T00:00:00,golf world rankings,[{'rendered_context': '--- ARTICLES [1] Player...,41c1a085-7b54-4a59-b7c8-145a2c9f225f,c96f580b-8dff-42a0-90dc-3e5c888680c6,693496.445
3,093a04a1-03f4-429c-a8cd-c2f7c8ea098c,True,Will Jordan Smith win the 2025 Italian Open?,2025-06-30T00:00:00,2025-06-25T00:00:00,This question resolves to Yes if Jordan Smith ...,2025-06-25T00:00:00,0,binary,1.0,...,The 2025 Italian Open (golf) took place from J...,https://vertexaisearch.cloud.google.com/ground...,Title: 2025 Italian Open betting tips: Our exp...,https://www.todays-golfer.com/news-and-events/...,2025-06-25T00:00:00,European Tour golf,"[{'rendered_context': '', 'search_query': 'Jor...",cebd9023-809d-441b-9ef5-251381880f5c,20e82969-aad5-477a-8e2e-cd641f7d7eec,493786.983
4,24bfbd2f-7a6c-4383-82aa-94d73f429685,True,Will Eddie Pepperell win at least one tourname...,2025-11-30T00:00:00,2025-06-25T00:00:00,The question resolves to 'Yes' if Eddie Pepper...,2025-06-25T00:00:00,0,binary,0.9,...,"The close date is 2025-11-30, and the question...",https://vertexaisearch.cloud.google.com/ground...,Title: Eddie Pepperell feeling refreshed after...,https://www.europeantour.com/dpworld-tour/news...,2025-06-25T00:00:00,European Tour golf,"[{'rendered_context': '', 'search_query': 'Edd...",a43d140a-a541-4537-8aab-6523ecbf79ce,3fa59bd7-f4b8-4ea9-b5b2-d8cba7d5ce0d,512717.986


## Model Training

Fine-tune a forecasting model on your dataset. For production training, generate more questions (increase `max_questions` or run without limit). Our reference experiments used 3,178 questions—see [Golf-Forecaster Model](https://huggingface.co/LightningRodLabs/Golf-Forecaster) and [Golf-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/GolfForecasting) for details.

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [7]:
from lightningrod import TrainingConfig

config = TrainingConfig(
    base_model="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=50,
)
cost_estimate = lr.training.estimate_cost(config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.31
Effective steps: 11
Train tokens: 1,040,793
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display.

In [15]:
job = lr.training.run(config, dataset=train_dataset, name="Golf forecasting")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job: Golf forecasting                                                                                        │
│                                                                                                                 │
│    Reward: latest -0.9948  avg -0.8261  (11 steps)  (higher is better)                                          │
│                                                                                                                 │
│    Cost:  $0.19                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job 6c82c197-0627-4ee8-954c-d5ddb93e66f2 completed with status: COMPLETED
Trained model ID: checkpoint:6c82c197-0627-4ee8-954c-d5ddb93e66f2


## Inference with your trained model

Use `lr.predict()` to run inference with your trained model.

In [ ]:
print(lr.predict(job.model_id, "Will Scottie Scheffler win the 2026 Masters?"))

## Run evals on trained model

Run test evals on your trained model against the test dataset. The eval job runs the model on the dataset and reports metrics.

In [20]:
eval_job = lr.evals.run(model_id=job.model_id, dataset=test_dataset)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    ID: cd970c00-d5b9-4db3-ac1f-f4815960abb0                                                                     │
│    Model: checkpoint:6c82c197-0627-4ee8-954c-d5ddb93e66f2                                                       │
│    Dataset: 708f1623-6f06-4897-bb2e-dd58b7aebd45                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓                                                                    │
│  ┃ Metric              ┃    base ┃ trained ┃                                                                    │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩                                                                    │
│  │ brier_score         │  0.2784 │  0.2377 │                                                                    │
│  │ ece                 │  0.2207 │  0.1597 │                                                                    │
│  │ mean_reward         │ -0.9210 │ -0.8026 │                                                                    │
│  │ mean_valid_reward   │ -0.9210 │ -0.8026 │                                                                    │
│  │ n_samples           │     143 │     143 │                                                                    │
│  │ n_valid             │     143 │     143 │                                                                    │
│  │ parse_rate          │  1.0000 │  1.0000 │                                                                    │
│  │ total_cost          │  0.0084 │  0.0084 │                                                                    │
│  │ total_input_tokens  │  115968 │  115968 │                                                                    │
│  │ total_output_tokens │    1403 │    1416 │                                                                    │
│  └─────────────────────┴─────────┴─────────┘                                                                    │
│                                                                                                                 │
│    Cost:  $0.02                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> Note: the trained model checkpoint will only be available for 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.